# 🔧Preprocessing & Feature Engineering
> **Project:** Salary Range Prediction — NYC Job Postings  
> **Goal:** Clean the raw data, normalize salaries, engineer new features, encode categoricals, and save the processed dataset.

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
os.makedirs("../data/processed", exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

print("✓ Libraries loaded")

✓ Libraries loaded


## 2. Load Raw Data

In [2]:
FILE_PATH = "../data/raw/Jobs_NYC_Postings.csv"

df = pd.read_csv(FILE_PATH)
df_clean = df.copy()   # always work on a copy, never modify the original

print(f"Loaded: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
df_clean.head(3)

Loaded: 5,120 rows × 30 columns


,Job ID,Agency,Posting Type,# Of Positions,Business Title,Civil Service Title,Title Classification,Title Code No,Level,Job Category,...,Additional Information,To Apply,Hours/Shift,Work Location 1,Recruitment Contact,Residency Requirement,Posting Date,Post Until,Posting Updated,Process Date
0,623668,DEPT OF HEALTH/MENTAL HYGIENE,External,1,"Research Analyst, IMAT, Bureau of Maternal Inf...",CITY RESEARCH SCIENTIST,Non-Competitive-5,21744,2,Constituent Services & Community Programs Heal...,...,**IMPORTANT NOTES TO ALL CANDIDATES: Please n...,Apply online with a cover letter to https://a1...,NaN,NaN,NaN,New York City residency is generally required ...,01/23/2024,22-May-2024,01/23/2024,03/04/2024
1,608193,DEPT OF DESIGN & CONSTRUCTION,Internal,1,Deputy Director,ADMINISTRATIVE PROJECT MANAGER,Competitive-1,8300B,0,"Policy, Research & Analysis Public Safety, Ins...",...,Authorization to work in the United States is ...,"For City Employees, please go to Employee Self...",35 Hours,"30-30 Thomson Avenue, LIC, NY 11101",NaN,New York City Residency is not required for th...,10/03/2023,NaN,11/29/2023,03/04/2024
2,625094,DEPT OF DESIGN & CONSTRUCTION,Internal,1,Senior Project Manager,ADMINISTRATIVE PROJECT MANAGER,Competitive-1,8300B,0,"Engineering, Architecture, & Planning",...,NaN,NaN,NaN,NaN,NaN,New York City Residency is not required for th...,02/01/2024,NaN,02/01/2024,03/04/2024


## 3. Normalize Salaries to Annual Equivalent
> **Why:** Hourly and Daily salaries are on a completely different scale than Annual.  
> We convert everything to annual so all rows are comparable.
> - Hourly → × 2,080 (52 weeks × 40 hrs)
> - Daily  → × 260  (52 weeks × 5 days)

In [3]:
HOURS_PER_YEAR = 2080
DAYS_PER_YEAR  = 260

def to_annual(row):
    freq = row["Salary Frequency"]
    if freq == "Hourly":
        return row["Salary Range From"] * HOURS_PER_YEAR, row["Salary Range To"] * HOURS_PER_YEAR
    elif freq == "Daily":
        return row["Salary Range From"] * DAYS_PER_YEAR, row["Salary Range To"] * DAYS_PER_YEAR
    else:
        return row["Salary Range From"], row["Salary Range To"]

df_clean[["Salary From Annual", "Salary To Annual"]] = df_clean.apply(
    to_annual, axis=1, result_type="expand"
)

print("Salary normalization done.")
print(df_clean[["Salary Frequency","Salary Range From","Salary From Annual"]].head(10))

Salary normalization done.
  Salary Frequency  Salary Range From  Salary From Annual
0           Annual         82506.0000           82506.000
1           Annual         58682.0000           58682.000
2           Annual         58682.0000           58682.000
3           Annual         50972.0000           50972.000
4           Annual         92301.0000           92301.000
5           Annual        140000.0000          140000.000
6           Annual         56313.0000           56313.000
7           Annual         48170.0000           48170.000
8           Hourly            22.9267           47687.536
9           Annual         93288.0000           93288.000


In [4]:
# Verify: check stats per frequency after normalization
df_clean.groupby("Salary Frequency")[["Salary From Annual","Salary To Annual"]].describe().round(0)

Salary From Annual                                      \
                              count     mean      std      min      25%   
Salary Frequency                                                          
Annual                       4686.0  70605.0  25573.0      0.0  56625.0   
Daily                          50.0  84047.0  34620.0  21380.0  73237.0   
Hourly                        384.0  65726.0  38660.0  31200.0  33134.0   

                                              Salary To Annual           \
                      50%       75%       max            count     mean   
Salary Frequency                                                          
Annual            65000.0   81571.0  225000.0           4686.0  98305.0   
Daily             77542.0  111810.0  160742.0             50.0  89484.0   
Hourly            47688.0   93226.0  164798.0            384.0  73607.0   

                                                                          
                      std      min      25%      50%       75%       max  
Salary Frequency                                                          
Annual            39599.0  35252.0  69709.0  90363.0  117000.0  252165.0  
Daily             33018.0  21380.0  85218.0  90272.0  111810.0  160742.0  
Hourly            37789.0  31200.0  41392.0  62379.0   95898.0  205587.0

## 4. Remove Invalid Salary Records

In [5]:
before = len(df_clean)

# Remove $0 or near-zero salaries
df_clean = df_clean[df_clean["Salary From Annual"] > 1000]

# Remove records where min salary > max salary (data error)
df_clean = df_clean[df_clean["Salary From Annual"] <= df_clean["Salary To Annual"]]

after = len(df_clean)
print(f"Removed : {before - after} invalid records")
print(f"Remaining: {after:,} rows")

Removed : 23 invalid records
Remaining: 5,097 rows


## 5. Drop Non-Useful Columns
> Columns dropped because they are: 100% missing, >60% missing, free-text (not ML-ready), or admin-only fields.

In [6]:
COLS_TO_DROP = [
    "Recruitment Contact",       # 100% missing
    "Post Until",                # 70.6% missing
    "Work Location 1",           # 63.1% missing — duplicate of Work Location
    "Hours/Shift",               # 62.7% missing
    "Additional Information",    # free text
    "To Apply",                  # free text
    "Job Description",           # free text (too unstructured for basic ML)
    "Minimum Qual Requirements", # free text
    "Preferred Skills",          # free text
    "Title Code No",             # internal admin code
    "Process Date",              # admin date
    "Posting Date",              # admin date
    "Posting Updated",           # admin date
    "Salary Range From",         # replaced by Salary From Annual
    "Salary Range To",           # replaced by Salary To Annual
    "Salary Frequency",          # already encoded into annual columns
]

existing_drop = [c for c in COLS_TO_DROP if c in df_clean.columns]
df_clean.drop(columns=existing_drop, inplace=True)

print(f"Dropped {len(existing_drop)} columns.")
print(f"Remaining columns ({df_clean.shape[1]}):")
for col in df_clean.columns:
    print(f"  • {col}")

Dropped 16 columns.
Remaining columns (16):
  • Job ID
  • Agency
  • Posting Type
  • # Of Positions
  • Business Title
  • Civil Service Title
  • Title Classification
  • Level
  • Job Category
  • Full-Time/Part-Time indicator
  • Career Level
  • Work Location
  • Division/Work Unit
  • Residency Requirement
  • Salary From Annual
  • Salary To Annual


## 6. Handle Remaining Missing Values

In [7]:
# Check what's still missing
missing_after = df_clean.isnull().sum()
print("Remaining missing values:")
print(missing_after[missing_after > 0])

Remaining missing values:
Full-Time/Part-Time indicator    117
dtype: int64


In [8]:
# Fill Full-Time indicator — assume Full-Time for the few missing
df_clean["Full-Time/Part-Time indicator"] = df_clean["Full-Time/Part-Time indicator"].fillna("F")

# Confirm no missing values remain in key columns
print("Missing values after imputation:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print("\n✓ No critical missing values remain")

Missing values after imputation:
Series([], dtype: int64)

✓ No critical missing values remain


## 7. Feature Engineering
> Creating new, more informative features from existing columns.

### 7.1 Title Seniority — extracted from Business Title

In [9]:
def extract_seniority(title):
    title = str(title).upper()
    if any(w in title for w in ["EXECUTIVE", "COMMISSIONER", "CHIEF", "DEPUTY COMMISSIONER"]):
        return "Executive"
    elif any(w in title for w in ["DIRECTOR"]):
        return "Director"
    elif any(w in title for w in ["MANAGER", "SUPERVISOR", "SUPERINTENDENT"]):
        return "Manager"
    elif any(w in title for w in ["SENIOR", "SR.", "SR ", "PRINCIPAL", "LEAD"]):
        return "Senior"
    elif any(w in title for w in ["DEPUTY", "ASSOCIATE", "ASSISTANT"]):
        return "Mid"
    elif any(w in title for w in ["JUNIOR", "JR.", "JR ", "ENTRY", "TRAINEE", "INTERN"]):
        return "Junior"
    else:
        return "Standard"

df_clean["Title Seniority"] = df_clean["Business Title"].apply(extract_seniority)

print("Title Seniority distribution:")
print(df_clean["Title Seniority"].value_counts())

Title Seniority distribution:
Title Seniority
Standard     2419
Manager       812
Mid           511
Director      441
Senior        391
Executive     352
Junior        171
Name: count, dtype: int64


### 7.2 Borough — extracted from Work Location

In [10]:
def extract_borough(location):
    loc = str(location).upper()
    if any(w in loc for w in ["MANHATTAN", "NEW YORK NY", "NEW YORK, NY",
                               "CENTRE ST", "BROADWAY", "FULTON", "WALL ST",
                               "CHAMBERS", "LEXINGTON", "PARK AVE"]):
        return "Manhattan"
    elif any(w in loc for w in ["BROOKLYN", "BKLYN"]):
        return "Brooklyn"
    elif any(w in loc for w in ["QUEENS", "QNS", "FLUSHING", "JAMAICA",
                                 "CORONA", "JUNCTION BLVD", "OZONE PARK"]):
        return "Queens"
    elif any(w in loc for w in ["BRONX", "BX", "METROTECH"]):
        return "Bronx"
    elif any(w in loc for w in ["STATEN ISLAND", "SI "]):
        return "Staten Island"
    else:
        return "Other"

df_clean["Borough"] = df_clean["Work Location"].apply(extract_borough)

print("Borough distribution:")
print(df_clean["Borough"].value_counts())

Borough distribution:
Borough
Other            3157
Queens            787
Manhattan         621
Bronx             270
Brooklyn          255
Staten Island       7
Name: count, dtype: int64


### 7.3 Other New Features

In [11]:
# Salary spread (useful for understanding prediction difficulty)
df_clean["Salary Spread"] = df_clean["Salary To Annual"] - df_clean["Salary From Annual"]

# Binary flags
df_clean["Is Full Time"] = (df_clean["Full-Time/Part-Time indicator"] == "F").astype(int)
df_clean["Is External"]  = (df_clean["Posting Type"] == "External").astype(int)

print("New features created:")
print(f"  Salary Spread  → mean: ${df_clean['Salary Spread'].mean():,.0f},"
      f" max: ${df_clean['Salary Spread'].max():,.0f}")
print(f"  Is Full Time   → {df_clean['Is Full Time'].value_counts().to_dict()}")
print(f"  Is External    → {df_clean['Is External'].value_counts().to_dict()}")

New features created:
  Salary Spread  → mean: $25,510, max: $160,000
  Is Full Time   → {1: 4935, 0: 162}
  Is External    → {0: 2600, 1: 2497}


## 8. Encode Categorical Variables

In [12]:
# Ordinal encoding for Career Level — preserves meaningful order
career_order = {
    "Entry-Level": 1,
    "Student": 2,
    "Experienced (non-manager)": 3,
    "Manager": 4,
    "Executive": 5
}
df_clean["Career Level Encoded"] = df_clean["Career Level"].map(career_order).fillna(3).astype(int)

print("Career Level Encoded:")
print(df_clean[["Career Level","Career Level Encoded"]].drop_duplicates().sort_values("Career Level Encoded"))

Career Level Encoded:
                 Career Level  Career Level Encoded
6                 Entry-Level                     1
19                    Student                     2
0   Experienced (non-manager)                     3
24                    Manager                     4
51                  Executive                     5


In [13]:
# Label encoding for remaining categorical columns
label_cols = [
    "Job Category",
    "Agency",
    "Borough",
    "Title Seniority",
    "Title Classification",
    "Posting Type",
]

le = LabelEncoder()
for col in label_cols:
    if col in df_clean.columns:
        df_clean[col + " Encoded"] = le.fit_transform(df_clean[col].astype(str))
        print(f"  {col:35s} → {df_clean[col].nunique():>3} unique values encoded")

  Job Category                        → 183 unique values encoded
  Agency                              →  59 unique values encoded
  Borough                             →   6 unique values encoded
  Title Seniority                     →   7 unique values encoded
  Title Classification                →   5 unique values encoded
  Posting Type                        →   2 unique values encoded


## 9. Define Final Feature Set

In [14]:
FEATURE_COLS = [
    "Career Level Encoded",
    "Job Category Encoded",
    "Agency Encoded",
    "Borough Encoded",
    "Title Seniority Encoded",
    "Title Classification Encoded",
    "# Of Positions",
    "Level",
    "Is Full Time",
    "Is External",
]

TARGET_MIN = "Salary From Annual"
TARGET_MAX = "Salary To Annual"

# Verify all feature columns exist
FEATURE_COLS = [f for f in FEATURE_COLS if f in df_clean.columns]

print(f"Features ({len(FEATURE_COLS)}):")
for f in FEATURE_COLS:
    print(f"  • {f}")
print(f"\nTargets:")
print(f"  • {TARGET_MIN}")
print(f"  • {TARGET_MAX}")

Features (10):
  • Career Level Encoded
  • Job Category Encoded
  • Agency Encoded
  • Borough Encoded
  • Title Seniority Encoded
  • Title Classification Encoded
  • # Of Positions
  • Level
  • Is Full Time
  • Is External

Targets:
  • Salary From Annual
  • Salary To Annual


In [15]:
# Preview the final feature matrix
df_clean[FEATURE_COLS + [TARGET_MIN, TARGET_MAX]].head(10)

,Career Level Encoded,Job Category Encoded,Agency Encoded,Borough Encoded,Title Seniority Encoded,Title Classification Encoded,# Of Positions,Level,Is Full Time,Is External,Salary From Annual,Salary To Annual
0,3,105,22,3,6,3,1,2,1,1,82506.000,94882.0
1,3,167,20,4,0,0,1,0,1,0,58682.000,134570.0
2,3,123,20,4,3,0,1,0,1,0,58682.000,91405.0
3,3,135,18,3,6,0,1,2,1,1,50972.000,82730.0
4,3,105,22,3,0,3,1,3,1,1,92301.000,106146.0
5,3,171,56,2,3,4,1,0,1,0,140000.000,160000.0
6,1,151,21,4,6,0,2,0,1,0,56313.000,64760.0
7,1,76,33,3,3,3,1,0,1,0,48170.000,48170.0
8,1,171,40,2,6,3,1,0,0,0,47687.536,58240.0
9,3,182,34,0,5,0,1,2,1,1,93288.000,107281.0


In [16]:
# Final shape & stats
print(f"Final dataset shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"\nTarget variable stats:")
print(df_clean[[TARGET_MIN, TARGET_MAX]].describe().round(0).to_string())

Final dataset shape : 5,097 rows × 28 columns

Target variable stats:
       Salary From Annual  Salary To Annual
count              5097.0            5097.0
mean              70687.0           96198.0
std               26573.0           39836.0
min               21380.0           21380.0
25%               55816.0           67632.0
50%               64922.0           89003.0
75%               81571.0          115000.0
max              225000.0          252165.0


## 10. Save Cleaned Dataset

In [17]:
OUTPUT_PATH = "../data/processed/salary_cleaned.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"✓ Cleaned dataset saved → {OUTPUT_PATH}")
print(f"  Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

✓ Cleaned dataset saved → ../data/processed/salary_cleaned.csv
  Shape: 5,097 rows × 28 columns


## ✅ Preprocessing Complete
**What was done:**
- Normalized Hourly (×2,080) and Daily (×260) salaries to annual equivalent
- Removed 23 invalid records ($0 salary or min > max)
- Dropped 16 non-useful/high-missing columns
- Engineered 5 new features: `Title Seniority`, `Borough`, `Salary Spread`, `Is Full Time`, `Is External`
- Ordinal encoded `Career Level` (1→5)
- Label encoded 6 categorical columns
- Saved cleaned data to `data/processed/salary_cleaned.csv`